In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Normalize timestamps to UTC then convert to Asia/Shanghai (UTC+8)
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Aggregate metric.csv by (cmdb_id, kpi_name)
metric_stats = metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count='count',
    p50=lambda x: x.quantile(0.5),
    p90=lambda x: x.quantile(0.9),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# Aggregate trace.csv by (cmdb_id, trace_name)
trace_stats = trace.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    count='count',
    p50=lambda x: x.quantile(0.5),
    p90=lambda x: x.quantile(0.9),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# Aggregate log.csv by (cmdb_id, log_name)
log_stats = log.groupby(['cmdb_id', 'log_name'])['value'].agg(
    count='count',
    p50=lambda x: x.quantile(0.5),
    p90=lambda x: x.quantile(0.9),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# Prepare combined table
metric_comb = metric_stats.rename(columns={'kpi_name': 'name'})
metric_comb['source'] = 'metric'
trace_comb = trace_stats.rename(columns={'trace_name': 'name'})
trace_comb['source'] = 'trace'
log_comb = log_stats.rename(columns={'log_name': 'name'})
log_comb['source'] = 'log'

combined = pd.concat([metric_comb[['source','cmdb_id','name','count','p50','p90','p95']],
                      trace_comb[['source','cmdb_id','name','count','p50','p90','p95']],
                      log_comb[['source','cmdb_id','name','count','p50','p90','p95']]],
                     ignore_index=True)

# Sort by count descending and limit to top 20
combined_top = combined.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Display the top combined table (compact)
combined_top

```
Out[1]:
```
```python
# Summarize the previously computed aggregation results succinctly.
# Reuse kernel variables: metric_stats, trace_stats, log_stats, combined_top

metric_count = metric_stats.shape[0]
trace_count = trace_stats.shape[0]
log_count = log_stats.shape[0]

# Distribution of sources in the top combined table
top_source_counts = combined_top['source'].value_counts().to_dict()

# Top CMDB ids in the combined_top
top_cmdb = combined_top['cmdb_id'].value_counts().head(5).to_dict()

# Example top 3 entries (cmdb_id:name and median)
examples = []
for _, row in combined_top.head(3).iterrows():
    examples.append(f"{row['cmdb_id']} | {row['name'][:80]}... p50={row['p50']:.4g} count={int(row['count'])}")

summary = (
    f"Summary:\n"
    f"- Aggregated groups: metrics={metric_count}, traces={trace_count}, logs={log_count}.\n"
    f"- Top-20 combined groups are dominated by metrics: source counts in top-20 = {top_source_counts}.\n"
    f"- Most frequent CMDBs in top-20: {top_cmdb}.\n"
    f"- Note: top groups show series length = 31 samples (count) for the displayed rows.\n"
    f"- Example top entries: {'; '.join(examples)}"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

source   cmdb_id                                               name  count           p50           p90           p95
0   metric      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  8.265509e+08  8.265509e+08  8.265509e+08
1   metric   Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00
2   metric  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02
3   metric  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  0.000000e+00  0.000000e+00
4   metric  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00  0.000000e+00  0.000000e+00
5   metric  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00
6   metric  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  6.041000e+04  6.043200e+04  6.043250e+04
7   metric  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00
8   metric   Redis01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  3.000000e-01  4.000000e-01  4.000000e-01
9   metric   Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  4.000000e+00  4.000000e+00  4.000000e+00
10  metric  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00
11  metric  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00
12  metric  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00
13  metric  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00
14  metric  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00
15  metric  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00
16  metric  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02  1.000000e+02  1.000000e+02
17  metric  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  0.000000e+00  0.000000e+00  0.000000e+00
18  metric   Redis01  redis-Redis_6379_Redis  (mem_fragmentation_ratio)     31  6.900000e-01  7.100000e-01  7.200000e-01
19  metric   Redis01          redis-Redis_6379_Redis  (pubsub_channels)     31  0.000000e+00  0.000000e+00  0.000000e+00```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Define incident window in UTC+8
start = pd.Timestamp('2021-03-04 02:30:00', tz=tz)
end = pd.Timestamp('2021-03-04 03:00:00', tz=tz)

# Filter raw tables to the incident window (these vars exist from previous cell)
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()
log_window = log[(log['timestamp'] >= start) & (log['timestamp'] <= end)].copy()
error_logs_window = error_logs[(error_logs['timestamp'] >= start) & (error_logs['timestamp'] <= end)].copy()

def summarize_window(df_window, stats_df, name_col, source_label):
    # Join global p95 (precomputed) onto windowed rows
    merged = df_window.merge(
        stats_df[['cmdb_id', name_col, 'p95']],
        on=['cmdb_id', name_col],
        how='left'
    )
    # Mark anomalies: value >= global p95 (NaN p95 -> False)
    merged['is_anomaly'] = (merged['value'] >= merged['p95']).fillna(False)
    # Group aggregates
    g = merged.groupby(['cmdb_id', name_col], as_index=False)
    agg = g.agg(
        count_in_window=('value', 'count'),
        anomaly_count=('is_anomaly', 'sum'),
        max_value_in_window=('value', 'max'),
        p95=('p95', 'first')
    )
    # earliest anomaly timestamp (NaT if no anomalies)
    earliest = merged[merged['is_anomaly']].groupby(['cmdb_id', name_col], as_index=False)['timestamp'].min()
    earliest = earliest.rename(columns={'timestamp': 'earliest_anomaly_timestamp'})
    agg = agg.merge(earliest, on=['cmdb_id', name_col], how='left')
    # Normalize column names
    agg = agg.rename(columns={name_col: 'name'})
    agg['source'] = source_label
    # Reorder columns
    agg = agg[['source', 'cmdb_id', 'name', 'count_in_window', 'anomaly_count',
               'earliest_anomaly_timestamp', 'max_value_in_window', 'p95']]
    return agg

metric_window_agg = summarize_window(metric_window, metric_stats, 'kpi_name', 'metric')
trace_window_agg  = summarize_window(trace_window, trace_stats, 'trace_name', 'trace')
log_window_agg    = summarize_window(log_window, log_stats, 'log_name', 'log')

# Combine and sort as requested
combined_window_agg = pd.concat([metric_window_agg, trace_window_agg, log_window_agg], ignore_index=True)
# Sort by anomaly_count desc, then earliest_anomaly_timestamp asc (NaT will be last)
combined_window_agg_sorted = combined_window_agg.sort_values(
    by=['anomaly_count', 'earliest_anomaly_timestamp'],
    ascending=[False, True]
).reset_index(drop=True)

# Limit to top 20 rows for compact display
combined_window_top20 = combined_window_agg_sorted.head(20)

# Counts of rows in each filtered table
metric_window_count = metric_window.shape[0]
trace_window_count = trace_window.shape[0]
log_window_count = log_window.shape[0]
error_logs_window_count = error_logs_window.shape[0]

# Final compact outputs
combined_window_top20, (metric_window_count, trace_window_count, log_window_count, error_logs_window_count)

```
Out[2]:
```
```python
# Summarize the analysis results (use existing kernel variables from previous steps)
metric_cnt, trace_cnt, log_cnt, error_logs_cnt = (metric_window_count, trace_window_count, log_window_count, error_logs_window_count)

top20 = combined_window_top20  # from previous cell
affected_cmdbs = ", ".join(top20['cmdb_id'].unique()[:10])
example_metrics = "; ".join(top20['name'].unique()[:5])

summary = (
    "Summary of incident-window analysis (2021-03-04 02:30:00 to 2021-03-04 03:00:00 UTC+8):\n"
    f"- Filtered row counts: metric={metric_cnt}, trace={trace_cnt}, log={log_cnt}, error_logs={error_logs_cnt}.\n"
    "- Top aggregated groups (shown top-20) are all from metrics.\n"
    "- For each top metric group: count_in_window = 31 and anomaly_count = 31, meaning every sample in the window "
    "met or exceeded the group's global P95 threshold.\n"
    "- Earliest anomaly timestamp observed in these top groups: 2021-03-04 02:30:00+08.\n"
    f"- CMDBs represented in top-20: {affected_cmdbs}.\n"
    f"- Example metric names in top-20: {example_metrics}.\n"
    "- No error log entries found in the window (error_logs count = 0).\n"
    "Observation: The top anomalies are metric-based and pervasive (all samples in-window >= global P95) "
    "for the listed metric groups; further root-cause investigation should focus on these metric series and their hosts."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source  cmdb_id                                               name  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window           p95
0   metric     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31             31  2021-03-04 02:30:00+08:00         2.040136e+09  2.040136e+09
1   metric     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               31             31  2021-03-04 02:30:00+08:00         8.265509e+08  8.265509e+08
2   metric     IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31             31  2021-03-04 02:30:00+08:00         0.000000e+00  0.000000e+00
3   metric     IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31             31  2021-03-04 02:30:00+08:00         0.000000e+00  0.000000e+00
4   metric     IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31             31  2021-03-04 02:30:00+08:00         3.000000e-01  3.000000e-01
5   metric     IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31             31  2021-03-04 02:30:00+08:00         2.040136e+09  2.040136e+09
6   metric     IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               31             31  2021-03-04 02:30:00+08:00         8.242818e+08  8.242818e+08
7   metric     IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31             31  2021-03-04 02:30:00+08:00         1.000000e+00  1.000000e+00
8   metric     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31             31  2021-03-04 02:30:00+08:00         3.000000e+00  3.000000e+00
9   metric     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31             31  2021-03-04 02:30:00+08:00         2.040136e+09  2.040136e+09
10  metric     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31             31  2021-03-04 02:30:00+08:00         5.060958e+08  5.060958e+08
11  metric     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31             31  2021-03-04 02:30:00+08:00         2.040136e+09  2.040136e+09
12  metric     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31             31  2021-03-04 02:30:00+08:00         6.889314e+08  6.889314e+08
13  metric  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31             31  2021-03-04 02:30:00+08:00         0.000000e+00  0.000000e+00
14  metric  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31             31  2021-03-04 02:30:00+08:00         0.000000e+00  0.000000e+00
15  metric  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31             31  2021-03-04 02:30:00+08:00         0.000000e+00  0.000000e+00
16  metric  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31             31  2021-03-04 02:30:00+08:00         7.000000e+00  7.000000e+00
17  metric  Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31             31  2021-03-04 02:30:00+08:00         0.000000e+00  0.000000e+00
18  metric  Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31             31  2021-03-04 02:30:00+08:00         0.000000e+00  0.000000e+00
19  metric  Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31             31  2021-03-04 02:30:00+08:00         0.000000e+00  0.000000e+00, (27408, 3759, 379, 0))```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables:
# metric_window (filtered rows in incident window), metric_stats (global p95 per group)

# Merge windowed rows with global p95
m_merged = metric_window.merge(
    metric_stats[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
).copy()

# Ensure timestamps are tz-aware (should already be), but enforce tz to be safe
if m_merged['timestamp'].dt.tz is None:
    m_merged['timestamp'] = m_merged['timestamp'].dt.tz_localize(tz)
else:
    m_merged['timestamp'] = m_merged['timestamp'].dt.tz_convert(tz)

# Compute boolean indicators (handle NaN p95 -> treat as non-anomaly)
m_merged['p95_is_na'] = m_merged['p95'].isna()
m_merged['gt_p95'] = (~m_merged['p95_is_na']) & (m_merged['value'] > m_merged['p95'])
m_merged['eq_p95'] = (~m_merged['p95_is_na']) & (m_merged['value'] == m_merged['p95'])

# Aggregate per metric group
grp = m_merged.groupby(['cmdb_id', 'kpi_name'], as_index=False)

metric_groups = grp.agg(
    count_in_window=('value', 'count'),
    count_gt_p95=('gt_p95', 'sum'),
    count_eq_p95=('eq_p95', 'sum'),
    max_value_in_window=('value', 'max'),
    p95=('p95', 'first')
)

# earliest timestamp where value > p95
earliest_gt = m_merged[m_merged['gt_p95']].groupby(['cmdb_id', 'kpi_name'], as_index=False)['timestamp'].min()
earliest_gt = earliest_gt.rename(columns={'timestamp': 'earliest_timestamp_of_value_gt_p95'})

metric_groups = metric_groups.merge(earliest_gt, on=['cmdb_id', 'kpi_name'], how='left')

# Compute relative_breach with p95==0 handling
def compute_relative_breach(row):
    p95 = row['p95']
    mx = row['max_value_in_window']
    if pd.isna(p95):
        return np.nan
    if p95 == 0:
        if mx > 0:
            return np.inf  # sentinel for infinite relative breach (report explicitly)
        else:
            return 0.0
    return (mx - p95) / p95

metric_groups['relative_breach'] = metric_groups.apply(compute_relative_breach, axis=1)

# Rename kpi_name to name for output and reorder columns
metric_groups = metric_groups.rename(columns={'kpi_name': 'name'})
metric_groups = metric_groups[[
    'cmdb_id', 'name', 'count_in_window', 'count_gt_p95', 'count_eq_p95',
    'max_value_in_window', 'p95', 'relative_breach', 'earliest_timestamp_of_value_gt_p95'
]]

# (A) Top 20 metric groups sorted by relative_breach desc (inf will appear first), then earliest timestamp asc
metric_groups_A = metric_groups.sort_values(
    by=['relative_breach', 'earliest_timestamp_of_value_gt_p95'],
    ascending=[False, True]
).reset_index(drop=True).head(20)

# (B) Aggregate by component (cmdb_id)
grp2 = metric_groups.groupby('cmdb_id', as_index=False)

metric_groups_B = grp2.agg(
    number_of_metric_groups_with_count_gt_p95_gt0=('count_gt_p95', lambda x: (x > 0).sum()),
    max_relative_breach_among_groups=('relative_breach', 'max'),
    earliest_timestamp_of_any_gt_p95=('earliest_timestamp_of_value_gt_p95', 'min')
)

# Sort B by max_relative_breach desc, then earliest timestamp asc, limit top 20
metric_groups_B = metric_groups_B.sort_values(
    by=['max_relative_breach_among_groups', 'earliest_timestamp_of_any_gt_p95'],
    ascending=[False, True]
).reset_index(drop=True).head(20)

# Total number of metric groups that had any sample value > p95
total_groups_with_gt_p95 = int((metric_groups['count_gt_p95'] > 0).sum())

# Compact outputs
metric_groups_A, metric_groups_B, total_groups_with_gt_p95

```
Out[3]:
```
```python
# Summarize the computed results in plain English using existing kernel variables:
# metric_groups_A (top metric groups by relative_breach), metric_groups_B (per-component aggregates),
# total_groups_with_gt_p95 (int)

top_groups = metric_groups_A.head(5)
top_components = metric_groups_B.head(5)
total_breached_groups = total_groups_with_gt_p95

examples = []
for _, r in top_groups.iterrows():
    rb = "inf" if r['relative_breach'] == float('inf') else f"{r['relative_breach']:.3g}"
    ts = (r['earliest_timestamp_of_value_gt_p95'].isoformat()
          if pd.notna(r['earliest_timestamp_of_value_gt_p95']) else "null")
    examples.append(f"{r['cmdb_id']} | {r['name'][:60]}... | max={r['max_value_in_window']:.6g} p95={r['p95']:.6g} rel_breach={rb} earliest={ts}")

components = []
for _, r in top_components.iterrows():
    rb = "inf" if r['max_relative_breach_among_groups'] == float('inf') else f"{r['max_relative_breach_among_groups']:.3g}"
    ts = (r['earliest_timestamp_of_any_gt_p95'].isoformat()
          if pd.notna(r['earliest_timestamp_of_any_gt_p95']) else "null")
    components.append(f"{r['cmdb_id']} | groups_with_gt_p95={int(r['number_of_metric_groups_with_count_gt_p95_gt0'])} | max_rel_breach={rb} earliest={ts}")

summary = (
    "Incident-window metric breach summary (2021-03-04 02:30–03:00 UTC+8):\n"
    f"- Total distinct metric groups with any sample > p95: {total_breached_groups}.\n"
    "- Top metric-group findings (examples, highest relative breach first):\n"
    + "\n".join(f"  - {e}" for e in examples) + "\n"
    "- Note: Several top groups show relative_breach = inf because their global p95 == 0 while max_value_in_window > 0.\n"
    "- Top impacted components (summary):\n"
    + "\n".join(f"  - {c}" for c in components) + "\n"
    "- Earliest observed breaches among top groups occurred at ~2021-03-04 02:30:00+08.\n"
    "Interpretation: Many metric groups across multiple components exceeded their historical P95 during the window. "
    "Cases with p95==0 should be investigated separately (these produce infinite relative breaches). "
    "Focus first on components with many breached groups (e.g., Mysql02, Tomcat02, Tomcat04) and the example metric groups listed above."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                               name  count_in_window  count_gt_p95  count_eq_p95  max_value_in_window        p95  relative_breach earliest_timestamp_of_value_gt_p95
0   Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               21             1            20             4.000000   0.000000              inf          2021-03-04 02:30:00+08:00
1    Mysql02           Mysql-MySQL_3306_GetResponseTimeOfMysqld               21             1            20             0.001000   0.000000              inf          2021-03-04 02:40:00+08:00
2   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               31             1            30             1.000000   0.000000              inf          2021-03-04 02:52:00+08:00
3    Mysql02        Mysql-MySQL_3306_Innodb data pending writes               20             1             0             1.000000   0.050000        19.000000          2021-03-04 02:30:00+08:00
4    Mysql02        Mysql-MySQL_3306_Innodb data pending fsyncs               20             1             0             1.000000   0.050000        19.000000          2021-03-04 02:35:00+08:00
5    Mysql02      Mysql-MySQL_3306_Innodb os log pending fsyncs               20             1             0             1.000000   0.050000        19.000000          2021-03-04 02:35:00+08:00
6    Mysql02                Mysql-MySQL_3306_MaxTrxRowsModified               20             1             0             1.000000   0.050000        19.000000          2021-03-04 02:35:00+08:00
7    Mysql02         Mysql-MySQL_3306_max trx lock memory bytes               20             1             0          1136.000000  56.800000        19.000000          2021-03-04 02:35:00+08:00
8    Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty               20             1             0             4.000000   0.200000        19.000000          2021-03-04 02:53:00+08:00
9   Tomcat01                             OSLinux-CPU_CPU_CPUWio               21             1             1             0.622700   0.033200        17.756024          2021-03-04 02:30:00+08:00
10      IG02                             OSLinux-CPU_CPU_CPUWio               31             2             0             0.457114   0.056050         7.155468          2021-03-04 02:31:00+08:00
11  apache01                             OSLinux-CPU_CPU_CPUWio               31             2             0             0.161900   0.022832         6.090770          2021-03-04 02:32:00+08:00
12  Tomcat04                             OSLinux-CPU_CPU_CPUWio               22             1             3             0.124700   0.020800         4.995192          2021-03-04 02:46:00+08:00
13   Redis01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps               16             1             0             0.100000   0.025000         3.000000          2021-03-04 02:30:00+08:00
14   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps               16             1             0             1.000000   0.250000         3.000000          2021-03-04 02:30:00+08:00
15   Redis01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               16             1             0            21.466700   5.366675         3.000000          2021-03-04 02:30:00+08:00
16   Redis01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps               16             1             0             0.083300   0.020825         3.000000          2021-03-04 02:30:00+08:00
17   Redis01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead               16             1             0            21.333300   5.333325         3.000000          2021-03-04 02:30:00+08:00
18      IG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...               16             1             0             0.300000   0.075000         3.000000          2021-03-04 02:44:00+08:00
19  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps               16             1             0             5.000000   1.250000         3.000000          2021-03-04 02:56:00+08:00,           cmdb_id  number_of_metric_groups_with_count_gt_p95_gt0  max_relative_breach_among_groups earliest_timestamp_of_any_gt_p95
0         Mysql02                                             78                               inf        2021-03-04 02:30:00+08:00
1        Tomcat02                                             39                               inf        2021-03-04 02:30:00+08:00
2        Tomcat04                                             35                               inf        2021-03-04 02:30:00+08:00
3         Mysql01                                             54                         19.000000        2021-03-04 02:30:00+08:00
4        Tomcat01                                             32                         17.756024        2021-03-04 02:30:00+08:00
5            IG02                                             34                          7.155468        2021-03-04 02:30:00+08:00
6        apache01                                             25                          6.090770        2021-03-04 02:30:00+08:00
7         Redis01                                             38                          3.000000        2021-03-04 02:30:00+08:00
8        apache02                                             24                          3.000000        2021-03-04 02:30:00+08:00
9            IG01                                             32                          2.333333        2021-03-04 02:30:00+08:00
10        Redis02                                             29                          1.183019        2021-03-04 02:30:00+08:00
11           MG01                                             33                          1.000000        2021-03-04 02:30:00+08:00
12           MG02                                             37                          1.000000        2021-03-04 02:30:00+08:00
13       Tomcat03                                             34                          1.000000        2021-03-04 02:30:00+08:00
14  ServiceTest10                                              1                          0.296992        2021-03-04 02:50:00+08:00
15   ServiceTest5                                              2                          0.200000        2021-03-04 02:52:00+08:00
16       dockerB1                                              4                          0.147059        2021-03-04 02:36:00+08:00
17   ServiceTest4                                              2                          0.129708        2021-03-04 02:42:00+08:00
18   ServiceTest9                                              2                          0.120000        2021-03-04 02:41:00+08:00
19   ServiceTest2                                              2                          0.111111        2021-03-04 02:34:00+08:00, 551)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: trace_window, trace_stats

# Merge trace_window with global p95 from trace_stats
t_merged = trace_window.merge(
    trace_stats[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
).copy()

# Ensure timestamps are timezone-aware in Asia/Shanghai
if t_merged['timestamp'].dt.tz is None:
    t_merged['timestamp'] = t_merged['timestamp'].dt.tz_localize(tz)
else:
    t_merged['timestamp'] = t_merged['timestamp'].dt.tz_convert(tz)

# Flags for breach comparisons (treat NaN p95 as non-breach)
t_merged['p95_is_na'] = t_merged['p95'].isna()
t_merged['gt_p95'] = (~t_merged['p95_is_na']) & (t_merged['value'] > t_merged['p95'])
t_merged['eq_p95'] = (~t_merged['p95_is_na']) & (t_merged['value'] == t_merged['p95'])

# Aggregate per trace group
grp = t_merged.groupby(['cmdb_id', 'trace_name'], as_index=False)
trace_groups = grp.agg(
    count_in_window=('value', 'count'),
    count_gt_p95=('gt_p95', 'sum'),
    count_eq_p95=('eq_p95', 'sum'),
    max_value_in_window=('value', 'max'),
    p95=('p95', 'first')
)

# earliest timestamp where value > p95
earliest_gt = t_merged[t_merged['gt_p95']].groupby(['cmdb_id', 'trace_name'], as_index=False)['timestamp'].min()
earliest_gt = earliest_gt.rename(columns={'timestamp': 'earliest_anomaly_timestamp'})

trace_groups = trace_groups.merge(earliest_gt, on=['cmdb_id', 'trace_name'], how='left')

# Compute relative_breach with p95==0 handling (inf when p95==0 and max>0)
def rel_breach(row):
    p = row['p95']
    m = row['max_value_in_window']
    if pd.isna(p):
        return np.nan
    if p == 0:
        return np.inf if (m > 0) else 0.0
    return (m - p) / p

trace_groups['relative_breach'] = trace_groups.apply(rel_breach, axis=1)

# Rename trace_name->name for output
trace_groups = trace_groups.rename(columns={'trace_name': 'name'})

# (1) Top 20 trace groups sorted by count_gt_p95 desc, then relative_breach desc
trace_groups_top20 = trace_groups.sort_values(
    by=['count_gt_p95', 'relative_breach'],
    ascending=[False, False]
).reset_index(drop=True).head(20)

# (A) number of distinct trace groups with any sample > p95
num_trace_groups_with_gt_p95 = int((trace_groups['count_gt_p95'] > 0).sum())

# (B) list of unique cmdb_id values among those trace groups (limit top 20)
trace_groups_cmdbs = (
    trace_groups[trace_groups['count_gt_p95'] > 0]
    .loc[:, ['cmdb_id']]
    .drop_duplicates()
    .reset_index(drop=True)
    .head(20)
)

# Second view restricted to previously top metric-impacted components
top_metric_components = ["Mysql02","Tomcat02","Tomcat04","Mysql01","Tomcat01","IG02","apache01","Redis01"]
trace_groups_restricted = trace_groups[trace_groups['cmdb_id'].isin(top_metric_components)].copy()

trace_groups_restricted_top20 = trace_groups_restricted.sort_values(
    by=['count_gt_p95', 'relative_breach'],
    ascending=[False, False]
).reset_index(drop=True).head(20)

# Compact outputs
trace_groups_top20, num_trace_groups_with_gt_p95, trace_groups_cmdbs, trace_groups_restricted_top20

```
Out[4]:
```
```python
# Summarize trace analysis results in plain English using existing kernel variables:
# trace_groups_top20, num_trace_groups_with_gt_p95, trace_groups_cmdbs, trace_groups_restricted_top20

# Prepare top examples from overall trace groups
top_overall = trace_groups_top20.head(6)
examples_overall = []
for _, r in top_overall.iterrows():
    examples_overall.append(
        f"{r['cmdb_id']} | {r['name']} | count_in_window={int(r['count_in_window'])} "
        f"count_gt_p95={int(r['count_gt_p95'])} earliest={r['earliest_anomaly_timestamp'].isoformat()} "
        f"rel_breach={r['relative_breach']:.3f}"
    )

# Prepare cmdb list (limited)
cmdb_list = ", ".join(trace_groups_cmdbs['cmdb_id'].astype(str).tolist())

# Prepare restricted-view examples for top metric-impacted components
restricted_examples = []
for _, r in trace_groups_restricted_top20.head(6).iterrows():
    restricted_examples.append(
        f"{r['cmdb_id']} | {r['name']} | count_in_window={int(r['count_in_window'])} "
        f"count_gt_p95={int(r['count_gt_p95'])} earliest={r['earliest_anomaly_timestamp'].isoformat()} "
        f"rel_breach={r['relative_breach']:.3f}"
    )

summary = (
    "Trace-event analysis summary (incident window 2021-03-04 02:30–03:00 UTC+8):\n\n"
    f"1) Top anomalous trace groups (examples):\n"
    + "\n".join(f"   - {e}" for e in examples_overall) + "\n\n"
    f"   Observations: These top groups each show a small number of breaches in-window (typically 2 samples > p95), "
    "with relative breaches ranging from ~0.28 up to ~1.27. Earliest anomaly timestamps are around 02:33–02:36+08.\n\n"
    f"2) Scope of impact:\n"
    f"   - Number of distinct trace groups with any sample > p95: {num_trace_groups_with_gt_p95}.\n"
    f"   - Example components involved (limited): {cmdb_list}.\n\n"
    f"3) Focused view on previously top metric-impacted components "
    "(Mysql02, Tomcat02, Tomcat04, Mysql01, Tomcat01, IG02, apache01, Redis01):\n"
    + "\n".join(f"   - {e}" for e in restricted_examples) + "\n\n"
    "Interpretation and next steps:\n"
    "- The trace anomalies point to increased downstream latencies/row counts involving IG nodes, Tomcat services, and MG nodes.\n"
    "- Earliest anomalies cluster around IG01/IG02 interactions with Tomcat and MG services (timestamps ~02:33–02:36+08),\n"
    "  suggesting a downstream slowdown originating near the IG <-> Tomcat <-> MG paths.\n"
    "- Action: prioritize investigation of IG01/IG02 and Tomcat04/Tomcat01 call paths, then inspect MG01/MG02 and associated downstream systems.\n"
    "- Correlate these trace findings with the metric anomalies already identified on those components for root-cause confirmation."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                               name  count_in_window  count_gt_p95  count_eq_p95  max_value_in_window        p95 earliest_anomaly_timestamp  relative_breach
0       MG01           trace.self.duration_mean               26             2             0              0.51375   0.226213  2021-03-04 02:35:00+08:00         1.271095
1       MG02           trace.self.duration_mean               26             2             0              0.51800   0.228958  2021-03-04 02:35:00+08:00         1.262426
2       IG02           trace.self.duration_mean               27             2             0              0.52400   0.258117  2021-03-04 02:33:00+08:00         1.030089
3   Tomcat01           trace.self.duration_mean               23             2             0              0.13100   0.075752  2021-03-04 02:36:00+08:00         0.729325
4       IG01    trace.to_Tomcat04.duration_mean               23             2             0              0.52300   0.304200  2021-03-04 02:35:00+08:00         0.719264
5   Tomcat04      trace.from_IG01.duration_mean               23             2             0              0.52300   0.304200  2021-03-04 02:35:00+08:00         0.719264
6       MG02  trace.from_Tomcat04.duration_mean               22             2             0              0.51300   0.303808  2021-03-04 02:35:00+08:00         0.688565
7   Tomcat04        trace.to_MG02.duration_mean               22             2             0              0.51300   0.303808  2021-03-04 02:35:00+08:00         0.688565
8   dockerB2               trace.self.row_count               22             2             0             90.00000  57.750000  2021-03-04 02:37:00+08:00         0.558442
9       IG01           trace.self.duration_mean               27             2             0              0.74150   0.488086  2021-03-04 02:36:00+08:00         0.519200
10      IG02     trace.to_Tomcat03.duration_p95               22             2             0              0.45025   0.299237  2021-03-04 02:34:00+08:00         0.504658
11  Tomcat03       trace.from_IG02.duration_p95               22             2             0              0.45025   0.299237  2021-03-04 02:34:00+08:00         0.504658
12      IG01     trace.to_Tomcat04.duration_p95               23             2             0              0.52300   0.358140  2021-03-04 02:35:00+08:00         0.460323
13  Tomcat04       trace.from_IG01.duration_p95               23             2             0              0.52300   0.358140  2021-03-04 02:35:00+08:00         0.460323
14      MG02      trace.from_Tomcat04.row_count               22             2             0              7.00000   4.950000  2021-03-04 02:48:00+08:00         0.414141
15  Tomcat04            trace.to_MG02.row_count               22             2             0              7.00000   4.950000  2021-03-04 02:48:00+08:00         0.414141
16      IG02        trace.to_Tomcat03.row_count               22             2             0              8.00000   5.900000  2021-03-04 02:34:00+08:00         0.355932
17  Tomcat03          trace.from_IG02.row_count               22             2             0              8.00000   5.900000  2021-03-04 02:34:00+08:00         0.355932
18      MG01  trace.from_Tomcat03.duration_mean               22             2             0              0.38950   0.303610  2021-03-04 02:33:00+08:00         0.282896
19  Tomcat03        trace.to_MG01.duration_mean               22             2             0              0.38950   0.303610  2021-03-04 02:33:00+08:00         0.282896, 205,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4   Tomcat01
5   Tomcat02
6   Tomcat03
7   Tomcat04
8   dockerA1
9   dockerA2
10  dockerB1
11  dockerB2,      cmdb_id                             name  count_in_window  count_gt_p95  count_eq_p95  max_value_in_window        p95 earliest_anomaly_timestamp  relative_breach
0       IG02         trace.self.duration_mean               27             2             0             0.524000   0.258117  2021-03-04 02:33:00+08:00         1.030089
1   Tomcat01         trace.self.duration_mean               23             2             0             0.131000   0.075752  2021-03-04 02:36:00+08:00         0.729325
2   Tomcat04    trace.from_IG01.duration_mean               23             2             0             0.523000   0.304200  2021-03-04 02:35:00+08:00         0.719264
3   Tomcat04      trace.to_MG02.duration_mean               22             2             0             0.513000   0.303808  2021-03-04 02:35:00+08:00         0.688565
4       IG02   trace.to_Tomcat03.duration_p95               22             2             0             0.450250   0.299237  2021-03-04 02:34:00+08:00         0.504658
5   Tomcat04     trace.from_IG01.duration_p95               23             2             0             0.523000   0.358140  2021-03-04 02:35:00+08:00         0.460323
6   Tomcat04          trace.to_MG02.row_count               22             2             0             7.000000   4.950000  2021-03-04 02:48:00+08:00         0.414141
7       IG02      trace.to_Tomcat03.row_count               22             2             0             8.000000   5.900000  2021-03-04 02:34:00+08:00         0.355932
8       IG02             trace.self.row_count               27             2             0            38.000000  30.200000  2021-03-04 02:34:00+08:00         0.258278
9   Tomcat04          trace.self.duration_p95               25             2             0             0.334750   0.267200  2021-03-04 02:36:00+08:00         0.252807
10  Tomcat01    trace.from_IG01.duration_mean               22             2             0             0.884000   0.722375  2021-03-04 02:36:00+08:00         0.223741
11  Tomcat04         trace.self.duration_mean               25             2             0             0.064375   0.053705  2021-03-04 02:36:00+08:00         0.198678
12      IG02  trace.to_Tomcat03.duration_mean               22             2             0             0.299000   0.250225  2021-03-04 02:41:00+08:00         0.194925
13  Tomcat04             trace.self.row_count               25             2             0            72.000000  61.000000  2021-03-04 02:33:00+08:00         0.180328
14  Tomcat02             trace.self.row_count               24             2             0            75.000000  64.950000  2021-03-04 02:47:00+08:00         0.154734
15  Tomcat01          trace.self.duration_p95               23             2             0             0.535750   0.468860  2021-03-04 02:36:00+08:00         0.142665
16  Tomcat02         trace.self.duration_mean               24             2             0             0.042356   0.039030  2021-03-04 02:33:00+08:00         0.085228
17      IG02          trace.self.duration_p95               27             2             0             0.722000   0.666705  2021-03-04 02:33:00+08:00         0.082938
18  Tomcat02      trace.to_MG01.duration_mean               22             2             0             0.167500   0.156213  2021-03-04 02:42:00+08:00         0.072257
19  Tomcat04       trace.to_MG02.duration_p95               22             2             0             0.513000   0.484882  2021-03-04 02:35:00+08:00         0.057988)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidate components (unique)
candidates = ["Mysql02","Tomcat02","Tomcat04","Mysql01","Tomcat01","IG02","IG01","MG01","MG02","Tomcat03"]

# Ensure timestamps in log_window and error_logs_window are timezone-aware in UTC+8
for df in (log_window, error_logs_window):
    if 'timestamp' in df.columns:
        if df['timestamp'].dt.tz is None:
            df['timestamp'] = df['timestamp'].dt.tz_localize(tz)
        else:
            df['timestamp'] = df['timestamp'].dt.tz_convert(tz)

# 1) Aggregate log_window by (cmdb_id, log_name) for candidate components
log_filtered = log_window[log_window['cmdb_id'].isin(candidates)].copy()

log_agg = log_filtered.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    count_in_window=('value', 'count'),
    sum_value=('value', 'sum'),
    max_value=('value', 'max'),
    earliest_timestamp=('timestamp', 'min'),
    latest_timestamp=('timestamp', 'max')
)

# Sort by sum_value desc and limit to top 20, select requested columns
log_agg_top20 = log_agg.sort_values('sum_value', ascending=False).head(20).reset_index(drop=True)[
    ['cmdb_id', 'log_name', 'count_in_window', 'sum_value', 'max_value', 'earliest_timestamp', 'latest_timestamp']
]

# 2) From error_logs_window, filter to candidate components and return up to 20 rows sorted by timestamp asc
error_logs_filtered = error_logs_window[error_logs_window['cmdb_id'].isin(candidates)].copy()
error_logs_filtered = error_logs_filtered.sort_values('timestamp', ascending=True).reset_index(drop=True)
error_logs_filtered_head = error_logs_filtered.loc[:, ['timestamp', 'cmdb_id', 'message']].head(20)

# Compact outputs
log_agg_top20, error_logs_filtered_head

```
Out[5]:
```
```python
# Summarize log and error-log findings for the candidate components (incident window)
# Reuse existing kernel variables: log_agg_top20, error_logs_filtered_head

top_logs = log_agg_top20.head(3)
error_count_rows = error_logs_filtered_head.shape[0]

examples = []
for _, r in top_logs.iterrows():
    examples.append(
        f"{r['cmdb_id']} | {r['log_name']} | count={int(r['count_in_window'])} sum={r['sum_value']:.2f} "
        f"max={r['max_value']:.2f} earliest={r['earliest_timestamp'].isoformat()} latest={r['latest_timestamp'].isoformat()}"
    )

summary = (
    "Log & error-log summary for candidate components (2021-03-04 02:30–03:00 UTC+8):\n\n"
    "- Top aggregated log findings (by sum_value):\n"
    + "\n".join(f"  - {e}" for e in examples) + "\n\n"
    "- Notes:\n"
    "  - Tomcat01 and Tomcat02 show the highest log.row_count totals in-window (30 samples each; sums 2229 and 2143),\n"
    "    indicating elevated request/row activity or backlog during the incident (timestamps span 02:30–02:59+08).\n"
    "  - GC pause aggregates (log.gc_pause_max) are present for Tomcat01/02 with total sums ~153 and per-sample max ~5.13s;\n"
    "    these values are not extremely large per-sample but warrant attention relative to throughput.\n"
    "  - log.error_count for Tomcat nodes is zero in-window (no aggregated error-counts observed).\n"
    f"  - Error log entries filtered to these components: {error_count_rows} rows (none found in the incident window).\n\n"
    "Interpretation & immediate recommendation:\n"
    "  - Focus investigation on Tomcat01 and Tomcat02 request handling (high row_count) and correlate with traces/metrics.\n"
    "  - Although GC pause totals exist, they are not dominant; prioritize throughput/backpressure and request queueing checks.\n"
    "  - No error log messages were captured for these components in the window; continue metric/trace correlation for root cause."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  count_in_window  sum_value  max_value        earliest_timestamp          latest_timestamp
0   Tomcat01     log.row_count               30    2229.00      80.00 2021-03-04 02:30:00+08:00 2021-03-04 02:59:00+08:00
1   Tomcat02     log.row_count               30    2143.00      83.00 2021-03-04 02:30:00+08:00 2021-03-04 02:59:00+08:00
2   Tomcat01  log.gc_pause_max               30     153.30       5.13 2021-03-04 02:30:00+08:00 2021-03-04 02:59:00+08:00
3   Tomcat02  log.gc_pause_max               30     153.21       5.13 2021-03-04 02:30:00+08:00 2021-03-04 02:59:00+08:00
4   Tomcat03     log.row_count               24     107.00      11.00 2021-03-04 02:30:00+08:00 2021-03-04 02:58:00+08:00
5   Tomcat04     log.row_count               25     103.00      11.00 2021-03-04 02:30:00+08:00 2021-03-04 02:58:00+08:00
6   Tomcat04  log.gc_pause_max                2       0.15       0.10 2021-03-04 02:40:00+08:00 2021-03-04 02:54:00+08:00
7   Tomcat03  log.gc_pause_max                1       0.07       0.07 2021-03-04 02:54:00+08:00 2021-03-04 02:54:00+08:00
8   Tomcat02   log.error_count               30       0.00       0.00 2021-03-04 02:30:00+08:00 2021-03-04 02:59:00+08:00
9   Tomcat01   log.error_count               30       0.00       0.00 2021-03-04 02:30:00+08:00 2021-03-04 02:59:00+08:00
10  Tomcat03   log.error_count               24       0.00       0.00 2021-03-04 02:30:00+08:00 2021-03-04 02:58:00+08:00
11  Tomcat04   log.error_count               25       0.00       0.00 2021-03-04 02:30:00+08:00 2021-03-04 02:58:00+08:00, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: metric_window, metric_stats
# 1) Prepare merged windowed data for Mysql02
mysql02_window = metric_window[metric_window['cmdb_id'] == 'Mysql02'].copy()

# Ensure timestamps tz-aware
if not mysql02_window.empty:
    if mysql02_window['timestamp'].dt.tz is None:
        mysql02_window['timestamp'] = mysql02_window['timestamp'].dt.tz_localize(tz)
    else:
        mysql02_window['timestamp'] = mysql02_window['timestamp'].dt.tz_convert(tz)

# Merge with precomputed p95 (metric_stats)
mysql02_merged = mysql02_window.merge(
    metric_stats[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
).copy()

# Flags for > p95 (treat NaN p95 as non-breach)
mysql02_merged['gt_p95'] = (~mysql02_merged['p95'].isna()) & (mysql02_merged['value'] > mysql02_merged['p95'])
mysql02_merged['eq_p95'] = (~mysql02_merged['p95'].isna()) & (mysql02_merged['value'] == mysql02_merged['p95'])

# Aggregate per kpi_name where any sample > p95
grp = mysql02_merged.groupby('kpi_name', as_index=False)

mysql02_agg = grp.agg(
    count_in_window=('value', 'count'),
    count_gt_p95=('gt_p95', 'sum'),
    max_value_in_window=('value', 'max'),
    p95=('p95', 'first')
)

# earliest timestamp where value > p95
earliest_gt = mysql02_merged[mysql02_merged['gt_p95']].groupby('kpi_name', as_index=False)['timestamp'].min()
earliest_gt = earliest_gt.rename(columns={'timestamp': 'earliest_timestamp_where_value_gt_p95'})

mysql02_agg = mysql02_agg.merge(earliest_gt, on='kpi_name', how='left')

# Filter to only groups that had any sample > p95
mysql02_agg = mysql02_agg[mysql02_agg['count_gt_p95'] > 0].copy()

# Compute relative_breach with p95==0 handling (inf if p95==0 and max>0)
def compute_rel(row):
    p = row['p95']
    m = row['max_value_in_window']
    if pd.isna(p):
        return np.nan
    if p == 0:
        return np.inf if (m > 0) else 0.0
    return (m - p) / p

mysql02_agg['relative_breach'] = mysql02_agg.apply(compute_rel, axis=1)

# Mark disk/InnoDB-related KPIs
keywords = ['innodb', 'fsync', 'pending', 'disk', 'dsk']
mysql02_agg['disk_innodb_flag'] = mysql02_agg['kpi_name'].str.lower().apply(
    lambda s: any(kw in s for kw in keywords)
)

# Sort by relative_breach descending and limit to top 20
mysql02_agg = mysql02_agg.sort_values(by='relative_breach', ascending=False).reset_index(drop=True)
mysql02_agg_top20 = mysql02_agg.head(20).loc[:, [
    'kpi_name', 'count_in_window', 'count_gt_p95', 'earliest_timestamp_where_value_gt_p95',
    'p95', 'max_value_in_window', 'relative_breach', 'disk_innodb_flag'
]]

# 2) Summary numbers
total_distinct_metric_groups_with_value_gt_p95 = int(mysql02_agg.shape[0])
if total_distinct_metric_groups_with_value_gt_p95 > 0:
    earliest_overall_timestamp_among_those = mysql02_agg['earliest_timestamp_where_value_gt_p95'].min()
else:
    earliest_overall_timestamp_among_those = pd.NaT

# 3) For top 5 kpi_name by relative_breach, return up to 3 raw metric samples each:
top5 = mysql02_agg.sort_values(by='relative_breach', ascending=False).head(5)['kpi_name'].tolist()

context_rows = []
for kpi in top5:
    # rows for this kpi in chronological order
    rows = mysql02_merged[mysql02_merged['kpi_name'] == kpi].sort_values('timestamp').reset_index(drop=True)
    # find earliest index where gt_p95
    gt_idx = rows.index[rows['gt_p95']].tolist()
    if not gt_idx:
        continue
    start_idx = gt_idx[0]
    # take the earliest anomaly and the next two samples (if available)
    context = rows.iloc[start_idx:start_idx+3][['timestamp', 'cmdb_id', 'kpi_name', 'value']]
    context_rows.append(context)

# Concatenate context rows, limit to 15 rows total
if context_rows:
    mysql02_context_samples = pd.concat(context_rows, ignore_index=True).head(15)
else:
    mysql02_context_samples = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'kpi_name', 'value'])

# 4) Disk/InnoDB marking already included as disk_innodb_flag column in aggregated table

# Final compact outputs (as required): aggregated table, two summary numbers, contextual samples
mysql02_agg_top20, (total_distinct_metric_groups_with_value_gt_p95, earliest_overall_timestamp_among_those), mysql02_context_samples

```
Out[6]:
```
```python
# Summarize Mysql02 findings in plain English using existing kernel variables:
# mysql02_agg_top20, total_distinct_metric_groups_with_value_gt_p95, earliest_overall_timestamp_among_those, mysql02_context_samples

import numpy as np

top_rows = mysql02_agg_top20.head(5)
top_list = []
for _, r in top_rows.iterrows():
    rb = r['relative_breach']
    rb_str = "inf (p95==0)" if np.isinf(rb) else f"{rb:.3g}"
    top_list.append(f"{r['kpi_name']} -> rel_breach={rb_str}, p95={r['p95']}, max={r['max_value_in_window']} (earliest breach {r['earliest_timestamp_where_value_gt_p95']})")

summary = (
    "Mysql02 incident-window summary (2021-03-04 02:30–03:00 UTC+8):\n"
    f"- Total distinct metric groups that exceeded their global P95: {total_distinct_metric_groups_with_value_gt_p95}.\n"
    f"- Earliest breach observed among those groups: {earliest_overall_timestamp_among_those}.\n\n"
    "Top breached KPIs (by relative breach, examples):\n"
    + "\n".join(f"  - {line}" for line in top_list) + "\n\n"
    "Key observations:\n"
    "- Several high relative breaches involve InnoDB/disk-related metrics (kpi names containing 'Innodb', 'fsync', 'pending', 'DSK').\n"
    "- Notable high-impact metrics:\n"
    "  * Mysql-MySQL_3306_max trx lock memory bytes, Mysql-MySQL_3306_MaxTrxRowsModified, "
    "and multiple Mysql-MySQL_3306_Innodb (data pending writes / pending fsyncs / os log pending fsyncs) — each shows large breaches (~19x p95).\n"
    "- Mysql-MySQL_3306_GetResponseTimeOfMysqld shows an infinite relative breach because its p95==0 while max>0; the absolute values are very small (0.001) so treat this separately.\n\n"
    "Interpretation and next steps:\n"
    "- Root-cause focus: investigate disk I/O and InnoDB flush/fsync behavior on Mysql02 (pending writes/fsyncs and trx lock memory spikes).\n"
    "- Correlate these InnoDB/disk metric spikes with the contextual samples and with trace/lock activity to confirm whether disk contention or transaction lock growth caused downstream impact.\n\n"
    "Contextual examples (earliest breach sample + up to two following samples) for the top KPIs are available in the variable 'mysql02_context_samples'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  count_in_window  count_gt_p95 earliest_timestamp_where_value_gt_p95         p95  max_value_in_window  relative_breach  disk_innodb_flag
0            Mysql-MySQL_3306_GetResponseTimeOfMysqld               21             1             2021-03-04 02:40:00+08:00    0.000000             0.001000              inf             False
1          Mysql-MySQL_3306_max trx lock memory bytes               20             1             2021-03-04 02:35:00+08:00   56.800000          1136.000000        19.000000             False
2                 Mysql-MySQL_3306_MaxTrxRowsModified               20             1             2021-03-04 02:35:00+08:00    0.050000             1.000000        19.000000             False
3       Mysql-MySQL_3306_Innodb os log pending fsyncs               20             1             2021-03-04 02:35:00+08:00    0.050000             1.000000        19.000000              True
4         Mysql-MySQL_3306_Innodb data pending fsyncs               20             1             2021-03-04 02:35:00+08:00    0.050000             1.000000        19.000000              True
5         Mysql-MySQL_3306_Innodb data pending writes               20             1             2021-03-04 02:30:00+08:00    0.050000             1.000000        19.000000              True
6      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps               15             1             2021-03-04 02:31:00+08:00    1.800000             6.000000         2.333333              True
7                     Mysql-MySQL_3306_ThreadsRunning               20             1             2021-03-04 02:35:00+08:00    1.100000             3.000000         1.727273             False
8   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKPer...               15             1             2021-03-04 02:30:00+08:00    0.880000             2.000000         1.272727              True
9    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite               15             1             2021-03-04 02:34:00+08:00   37.839990            78.533300         1.075405              True
10                    Mysql-MySQL_3306_Handler Update               20             1             2021-03-04 02:35:00+08:00    0.105000             0.200000         0.904762             False
11                        Mysql-MySQL_3306_Com Update               20             1             2021-03-04 02:35:00+08:00    0.105000             0.200000         0.904762             False
12    Mysql-MySQL_3306_Innodb buffer pool pages dirty               20             1             2021-03-04 02:35:00+08:00   17.800000            33.000000         0.853933              True
13                         OSLinux-CPU_CPU_CPUSysTime               31             2             2021-03-04 02:34:00+08:00    0.513169             0.942965         0.837533             False
14     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps               10             1             2021-03-04 02:51:00+08:00    2.200000             4.000000         0.818182              True
15  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...               10             1             2021-03-04 02:51:00+08:00  207.800000           368.000000         0.770934              True
16    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               15             1             2021-03-04 02:50:00+08:00  219.973310           348.400000         0.583829              True
17                            OSLinux-CPU_CPU_CPULoad               31             2             2021-03-04 02:34:00+08:00    0.160000             0.250000         0.562500             False
18    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps               15             1             2021-03-04 02:34:00+08:00    1.571667             2.400000         0.527041              True
19              Mysql-MySQL_3306_Innodb pages created               20             1             2021-03-04 02:35:00+08:00    0.068365             0.100000         0.462737              True, (78, Timestamp('2021-03-04 02:30:00+0800', tz='Asia/Shanghai')),                    timestamp  cmdb_id                                       kpi_name     value
0  2021-03-04 02:40:00+08:00  Mysql02       Mysql-MySQL_3306_GetResponseTimeOfMysqld     0.001
1  2021-03-04 02:42:00+08:00  Mysql02       Mysql-MySQL_3306_GetResponseTimeOfMysqld     0.000
2  2021-03-04 02:43:00+08:00  Mysql02       Mysql-MySQL_3306_GetResponseTimeOfMysqld     0.000
3  2021-03-04 02:35:00+08:00  Mysql02     Mysql-MySQL_3306_max trx lock memory bytes  1136.000
4  2021-03-04 02:36:00+08:00  Mysql02     Mysql-MySQL_3306_max trx lock memory bytes     0.000
5  2021-03-04 02:38:00+08:00  Mysql02     Mysql-MySQL_3306_max trx lock memory bytes     0.000
6  2021-03-04 02:35:00+08:00  Mysql02            Mysql-MySQL_3306_MaxTrxRowsModified     1.000
7  2021-03-04 02:36:00+08:00  Mysql02            Mysql-MySQL_3306_MaxTrxRowsModified     0.000
8  2021-03-04 02:38:00+08:00  Mysql02            Mysql-MySQL_3306_MaxTrxRowsModified     0.000
9  2021-03-04 02:35:00+08:00  Mysql02  Mysql-MySQL_3306_Innodb os log pending fsyncs     1.000
10 2021-03-04 02:36:00+08:00  Mysql02  Mysql-MySQL_3306_Innodb os log pending fsyncs     0.000
11 2021-03-04 02:38:00+08:00  Mysql02  Mysql-MySQL_3306_Innodb os log pending fsyncs     0.000
12 2021-03-04 02:35:00+08:00  Mysql02    Mysql-MySQL_3306_Innodb data pending fsyncs     1.000
13 2021-03-04 02:36:00+08:00  Mysql02    Mysql-MySQL_3306_Innodb data pending fsyncs     0.000
14 2021-03-04 02:38:00+08:00  Mysql02    Mysql-MySQL_3306_Innodb data pending fsyncs     0.000)```
```